In [1]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from transformers import AutoTokenizer
import numpy as np
import os
import math
import random
from dataclasses import dataclass
from collections import defaultdict
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from torchcrf import CRF
import spacy
from tqdm.auto import tqdm
import re

/home/omer_ahmed/Experiments/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_data = pd.read_parquet("/home/omer_ahmed/Experiments/Span-models/processed_span_data/train.parquet")
val_data = pd.read_parquet("/home/omer_ahmed/Experiments/Span-models/processed_span_data/val.parquet")
train_data.drop(columns=["span_text"], inplace=True)
val_data.drop(columns=["span_text"], inplace=True)


In [6]:
val_data

,article_id,span_start,span_end,article_text
0,111111111,149,157,Next plague outbreak in Madagascar could be 's...
1,111111111,265,323,Next plague outbreak in Madagascar could be 's...
2,111111111,1069,1091,Next plague outbreak in Madagascar could be 's...
3,111111111,1334,1462,Next plague outbreak in Madagascar could be 's...
4,111111111,1577,1616,Next plague outbreak in Madagascar could be 's...
...,...,...,...,...
3170,11511,0,76,"But soon, Coleman would later tell the FBI, th..."
3171,11519,241,267,Coleman told the agent that “he was either cra...
3172,11523,0,24,“It 7 was a hell for me.”
3173,11530,83,116,"Trans men trying to conceive, as documented by..."


## Model Span Detection

In [ ]:
# =========================================================
# 1. CONFIG
# =========================================================
@dataclass
class CFG:
    model_name: str = "roberta-large"
    max_length: int = 512
    stride: int = 384              # 512 - 384 = 128 overlap
    batch_size: int = 2
    lr: float = 2e-5
    weight_decay: float = 0.01
    epochs: int = 3
    warmup_ratio: float = 0.1
    num_workers: int = 2
    seed: int = 42
    lstm_hidden: int = 512
    pos_dim: int = 0
    ner_dim: int = 0
    dropout: float = 0.3
    save_path: str = "best_span_roberta_no_pos_ner_2.pt"


# =========================================================
# 2. LABELS
# BIOES for propaganda span detection
# =========================================================
LABEL2ID = {
    "O": 0,
    "B-PROP": 1,
    "I-PROP": 2,
    "E-PROP": 3,
    "S-PROP": 4,
}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}
NUM_LABELS = len(LABEL2ID)


# =========================================================
# 3. REPRODUCIBILITY
# =========================================================
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


# =========================================================
# 4. ARTICLE-LEVEL RECORDS
# Convert row-per-span dataframe into one row per article
# =========================================================
def build_article_records(df: pd.DataFrame):
    records = []

    grouped = df.groupby("article_id", as_index=False)
    for _, g in grouped:
        article_id = str(g["article_id"].iloc[0])

        article_text = g["article_text"].iloc[0]
        article_text = "" if pd.isna(article_text) else str(article_text)

        spans = (
            g[["span_start", "span_end"]]
            .drop_duplicates()
            .sort_values(["span_start", "span_end"])
            .values.tolist()
        )

        clean_spans = []
        for s, e in spans:
            if pd.notna(s) and pd.notna(e):
                s, e = int(s), int(e)
                s = max(0, s)
                e = min(len(article_text), e)
                if s < e:
                    clean_spans.append((s, e))

        records.append({
            "article_id": article_id,
            "article_text": article_text,
            "gold_spans": clean_spans
        })

    return records
# =========================================================
# 5. SPACY-BASED POS / NER VOCAB BUILDING
# =========================================================
def build_pos_ner_vocab(article_records, nlp):
    """
    Build vocabularies from training articles only.
    0 is reserved for PAD/UNK.
    """
    pos_set = set()
    ner_set = set()

    for rec in tqdm(article_records, desc="Building POS/NER vocab"):
        doc = nlp(rec["article_text"])
        for tok in doc:
            pos_set.add(tok.pos_)
        for ent in doc.ents:
            ner_set.add(ent.label_)

    pos_vocab = {"<PAD>": 0}
    ner_vocab = {"<PAD>": 0}

    for i, tag in enumerate(sorted(pos_set), start=1):
        pos_vocab[tag] = i

    for i, tag in enumerate(sorted(ner_set), start=1):
        ner_vocab[tag] = i

    return pos_vocab, ner_vocab


# =========================================================
# 6. CHAR-LEVEL AUXILIARY MAPS
# POS / NER per character position
# =========================================================
def build_char_feature_maps(text, nlp, pos_vocab, ner_vocab):
    """
    Returns two arrays of length len(text):
      pos_char_ids[char_idx]
      ner_char_ids[char_idx]
    """
    pos_char_ids = np.zeros(len(text), dtype=np.int64)
    ner_char_ids = np.zeros(len(text), dtype=np.int64)

    doc = nlp(text)

    # POS from tokens
    for tok in doc:
        pos_id = pos_vocab.get(tok.pos_, 0)
        start, end = tok.idx, tok.idx + len(tok.text)
        pos_char_ids[start:end] = pos_id

    # NER from entities
    for ent in doc.ents:
        ner_id = ner_vocab.get(ent.label_, 0)
        start, end = ent.start_char, ent.end_char
        ner_char_ids[start:end] = ner_id

    return pos_char_ids, ner_char_ids


# =========================================================
# 7. GOLD CHAR MASK
# 1 for propaganda chars, 0 otherwise
# =========================================================
def build_gold_char_mask(text_len, spans):
    mask = np.zeros(text_len, dtype=np.int64)
    for s, e in spans:
        s = max(0, s)
        e = min(text_len, e)
        if s < e:
            mask[s:e] = 1
    return mask


# =========================================================
# 8. TOKEN TAGGING FROM CHAR SPANS
# Convert token offsets -> BIOES labels
# =========================================================
def assign_bioes_from_offsets(offsets, gold_char_mask):
    """
    offsets: list of (start, end) for tokens in one window
    gold_char_mask: array over full article chars

    Returns:
      labels: list[int] length = len(offsets)
      crf_mask: list[int] where 1 means real token, 0 means special/pad
    """
    token_is_prop = []
    crf_mask = []

    for start, end in offsets:
        if end <= start:
            token_is_prop.append(0)
            crf_mask.append(0)   # special token / padding
        else:
            overlap = gold_char_mask[start:end].max() > 0
            token_is_prop.append(1 if overlap else 0)
            crf_mask.append(1)

    labels = [LABEL2ID["O"]] * len(offsets)

    i = 0
    n = len(offsets)
    while i < n:
        if crf_mask[i] == 0 or token_is_prop[i] == 0:
            i += 1
            continue

        j = i
        while j + 1 < n and crf_mask[j + 1] == 1 and token_is_prop[j + 1] == 1:
            j += 1

        # span from i..j
        if i == j:
            labels[i] = LABEL2ID["S-PROP"]
        else:
            labels[i] = LABEL2ID["B-PROP"]
            for k in range(i + 1, j):
                labels[k] = LABEL2ID["I-PROP"]
            labels[j] = LABEL2ID["E-PROP"]

        i = j + 1
    # Ensure first timestep mask is valid for CRF
    if len(crf_mask) > 0:
        crf_mask[0] = 1
    return labels, crf_mask


# =========================================================
# 9. DATASET
# Sliding windows + POS/NER alignment + BIOES labels
# =========================================================
class PTCSpanDataset(Dataset):
    def __init__(self, article_records, tokenizer, nlp, pos_vocab, ner_vocab,
                 max_length=512, stride=384, is_train=True):
        self.examples = []
        self.tokenizer = tokenizer
        self.max_length = max_length
        self.stride = stride
        self.is_train = is_train

        for rec in tqdm(article_records, desc="Preparing dataset"):
            article_id = rec["article_id"]
            text = rec["article_text"]
            gold_spans = rec["gold_spans"]

            gold_char_mask = build_gold_char_mask(len(text), gold_spans)

            if nlp is not None:
                pos_char_ids, ner_char_ids = build_char_feature_maps(text, nlp, pos_vocab, ner_vocab)
            else:
                pos_char_ids = np.zeros(len(text), dtype=np.int64)
                ner_char_ids = np.zeros(len(text), dtype=np.int64)

            enc = tokenizer(
                text,
                return_offsets_mapping=True,
                return_overflowing_tokens=True,
                truncation=True,
                max_length=max_length,
                stride=max_length - stride,
                padding="max_length"
            )

            for i in range(len(enc["input_ids"])):
                input_ids = enc["input_ids"][i]
                attention_mask = enc["attention_mask"][i]
                offsets = enc["offset_mapping"][i]

                labels, crf_mask = assign_bioes_from_offsets(offsets, gold_char_mask)

                pos_ids = []
                ner_ids = []

                for start, end in offsets:
                    if end <= start:
                        pos_ids.append(0)
                        ner_ids.append(0)
                    else:
                        pos_ids.append(int(pos_char_ids[start]))
                        ner_ids.append(int(ner_char_ids[start]))

                self.examples.append({
                    "article_id": article_id,
                    "input_ids": torch.tensor(input_ids, dtype=torch.long),
                    "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
                    "crf_mask": torch.tensor(crf_mask, dtype=torch.bool),
                    "pos_ids": torch.tensor(pos_ids, dtype=torch.long),
                    "ner_ids": torch.tensor(ner_ids, dtype=torch.long),
                    "labels": torch.tensor(labels, dtype=torch.long),
                    "offset_mapping": offsets,
                    "text_len": len(text),
                })

    def __len__(self):
        return len(self.examples)

    def __getitem__(self, idx):
        ex = self.examples[idx]
        return {
            "article_id": ex["article_id"],
            "input_ids": ex["input_ids"],
            "attention_mask": ex["attention_mask"],
            "crf_mask": ex["crf_mask"],
            "pos_ids": ex["pos_ids"],
            "ner_ids": ex["ner_ids"],
            "labels": ex["labels"],
            "offset_mapping": ex["offset_mapping"],
            "text_len": ex["text_len"],
        }

def collate_fn(batch):
    return {
        "article_id": [x["article_id"] for x in batch],
        "input_ids": torch.stack([x["input_ids"] for x in batch]),
        "attention_mask": torch.stack([x["attention_mask"] for x in batch]),
        "crf_mask": torch.stack([x["crf_mask"] for x in batch]),
        "pos_ids": torch.stack([x["pos_ids"] for x in batch]),
        "ner_ids": torch.stack([x["ner_ids"] for x in batch]),
        "labels": torch.stack([x["labels"] for x in batch]),
        "offset_mapping": [x["offset_mapping"] for x in batch],
        "text_len": [x["text_len"] for x in batch],
    }


# =========================================================
# 10. MODEL
# DeBERTa + POS/NER fusion + BiLSTM + CRF
# Ready for future discourse_feats
# =========================================================
class DebertaSpanTagger(nn.Module):
    def __init__(
        self,
        model_name="roberta-large",
        num_labels=5,
        num_pos_tags=50,
        num_ner_tags=30,
        pos_dim=32,
        ner_dim=32,
        discourse_dim=0,
        lstm_hidden=512,
        dropout=0.1
    ):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        hidden_size = self.encoder.config.hidden_size

        self.pos_dim = pos_dim
        self.ner_dim = ner_dim
        self.discourse_dim = discourse_dim

        if pos_dim > 0:
            self.pos_embedding = nn.Embedding(num_pos_tags, pos_dim)
        else:
            self.pos_embedding = None

        if ner_dim > 0:
            self.ner_embedding = nn.Embedding(num_ner_tags, ner_dim)
        else:
            self.ner_embedding = None

        fusion_input_dim = hidden_size + pos_dim + ner_dim + discourse_dim
        self.fusion = nn.Linear(fusion_input_dim, hidden_size)

        self.bilstm = nn.LSTM(
            input_size=hidden_size,
            hidden_size=lstm_hidden // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(lstm_hidden, num_labels)
        self.crf = CRF(num_labels, batch_first=True)

        nn.init.xavier_uniform_(self.fusion.weight)
        nn.init.xavier_uniform_(self.classifier.weight)

    def forward(self, input_ids, attention_mask, crf_mask, pos_ids, ner_ids, discourse_feats=None, labels=None):
        enc = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        token_embeddings = enc.last_hidden_state

        if not torch.isfinite(token_embeddings).all():
            raise ValueError("Non-finite values detected in encoder output")

        feats = [token_embeddings]

        if self.pos_embedding is not None:
            pos_emb = self.pos_embedding(pos_ids)
            if not torch.isfinite(pos_emb).all():
                raise ValueError("Non-finite values detected in POS embeddings")
            feats.append(pos_emb)

        if self.ner_embedding is not None:
            ner_emb = self.ner_embedding(ner_ids)
            if not torch.isfinite(ner_emb).all():
                raise ValueError("Non-finite values detected in NER embeddings")
            feats.append(ner_emb)

        if discourse_feats is not None:
            if not torch.isfinite(discourse_feats).all():
                raise ValueError("Non-finite values detected in discourse features")
            feats.append(discourse_feats)

        fused = torch.cat(feats, dim=-1)

        if not torch.isfinite(fused).all():
            raise ValueError("Non-finite values detected after concatenation")

        fused = self.fusion(fused)

        if not torch.isfinite(fused).all():
            raise ValueError("Non-finite values detected after fusion layer")

        lstm_out, _ = self.bilstm(fused)

        if not torch.isfinite(lstm_out).all():
            raise ValueError("Non-finite values detected after BiLSTM")

        lstm_out = self.dropout(lstm_out)
        emissions = self.classifier(lstm_out)

        if not torch.isfinite(emissions).all():
            raise ValueError("Non-finite values detected in emissions")

        if labels is not None:
            loss = -self.crf(emissions, labels, mask=crf_mask, reduction="token_mean")
            if not torch.isfinite(loss):
                raise ValueError("Non-finite CRF loss detected")
            return loss
        else:
            preds = self.crf.decode(emissions, mask=crf_mask)
            return preds


# =========================================================
# 11. TRAIN / EVAL UTILITIES
# =========================================================
def token_level_f1(gold_list, pred_list):
    """
    Micro-F1 over non-padding real tokens.
    Binary evaluation:
      propaganda tags = {B,I,E,S}
      non-propaganda = O
    """
    tp = fp = fn = 0

    for gold_seq, pred_seq in zip(gold_list, pred_list):
        for g, p in zip(gold_seq, pred_seq):
            g_bin = 1 if g != LABEL2ID["O"] else 0
            p_bin = 1 if p != LABEL2ID["O"] else 0

            if g_bin == 1 and p_bin == 1:
                tp += 1
            elif g_bin == 0 and p_bin == 1:
                fp += 1
            elif g_bin == 1 and p_bin == 0:
                fn += 1

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1


def spans_from_char_mask(mask):
    spans = []
    in_span = False
    start = 0
    for i, val in enumerate(mask):
        if val == 1 and not in_span:
            start = i
            in_span = True
        elif val == 0 and in_span:
            spans.append((start, i))
            in_span = False
    if in_span:
        spans.append((start, len(mask)))
    return spans


def exact_span_f1(gold_spans_by_article, pred_spans_by_article):
    tp = fp = fn = 0

    all_articles = set(gold_spans_by_article.keys()) | set(pred_spans_by_article.keys())

    for aid in all_articles:
        gold = set(gold_spans_by_article.get(aid, []))
        pred = set(pred_spans_by_article.get(aid, []))

        tp += len(gold & pred)
        fp += len(pred - gold)
        fn += len(gold - pred)

    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

def decode_bioes_token_spans(tag_seq, offsets):
    """
    Convert one predicted BIOES tag sequence into character spans.

    Parameters
    ----------
    tag_seq : list[int]
        Predicted label ids for one window.
    offsets : list[tuple[int, int]]
        Offset mapping for one window.

    Returns
    -------
    spans : list[tuple[int, int]]
        Character spans decoded conservatively from valid BIOES structure.
    """
    spans = []
    i = 0
    n = len(tag_seq)

    while i < n:
        tag = tag_seq[i]

        # skip special/pad tokens
        if offsets[i][1] <= offsets[i][0]:
            i += 1
            continue

        # O
        if tag == LABEL2ID["O"]:
            i += 1
            continue

        # S-PROP => single-token span
        if tag == LABEL2ID["S-PROP"]:
            start, end = offsets[i]
            spans.append((start, end))
            i += 1
            continue

        # B-PROP => try to form B ... I* ... E
        if tag == LABEL2ID["B-PROP"]:
            start = offsets[i][0]
            j = i + 1

            # Case 1: immediate B E
            if j < n and offsets[j][1] > offsets[j][0] and tag_seq[j] == LABEL2ID["E-PROP"]:
                end = offsets[j][1]
                spans.append((start, end))
                i = j + 1
                continue

            # Case 2: B I* E
            while j < n:
                if offsets[j][1] <= offsets[j][0]:
                    break

                if tag_seq[j] == LABEL2ID["I-PROP"]:
                    j += 1
                    continue

                if tag_seq[j] == LABEL2ID["E-PROP"]:
                    end = offsets[j][1]
                    spans.append((start, end))
                    i = j + 1
                    break

                # broken chain
                break
            else:
                # fell off sequence without E
                pass

            if i < n and tag_seq[i] == LABEL2ID["B-PROP"]:
                # no valid closing E found, drop broken span
                i += 1
            continue

        # isolated I or E => ignore conservatively
        if tag in (LABEL2ID["I-PROP"], LABEL2ID["E-PROP"]):
            i += 1
            continue

        i += 1

    return spans
def merge_overlapping_spans(spans):
    """
    Merge overlapping or touching character spans.
    """
    if not spans:
        return []

    spans = sorted(spans, key=lambda x: (x[0], x[1]))
    merged = [spans[0]]

    for s, e in spans[1:]:
        last_s, last_e = merged[-1]

        if s <= last_e:  # overlap or touching
            merged[-1] = (last_s, max(last_e, e))
        else:
            merged.append((s, e))

    return merged
@torch.no_grad()
def predict_spans(model, loader, device):
    model.eval()

    pred_spans_by_article_raw = defaultdict(list)
    all_gold_token = []
    all_pred_token = []

    for batch in tqdm(loader, desc="Predicting"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        crf_mask = batch["crf_mask"].to(device)
        pos_ids = batch["pos_ids"].to(device)
        ner_ids = batch["ner_ids"].to(device)
        labels = batch["labels"].cpu().numpy()

        preds = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            crf_mask=crf_mask,
            pos_ids=pos_ids,
            ner_ids=ner_ids,
            labels=None
        )

        for i in range(len(preds)):
            article_id = batch["article_id"][i]
            offsets = batch["offset_mapping"][i]
            gold_seq = labels[i].tolist()
            pred_seq = preds[i]

            # token-level eval on real tokens only
            real_gold = []
            real_pred = []
            for g, p, (s, e) in zip(gold_seq, pred_seq, offsets):
                if e > s:
                    real_gold.append(g)
                    real_pred.append(p)

            all_gold_token.append(real_gold)
            all_pred_token.append(real_pred)

            # Proper BIOES decoding
            window_spans = decode_bioes_token_spans(pred_seq, offsets)
            pred_spans_by_article_raw[article_id].extend(window_spans)

    # Merge overlapping spans from different windows of same article
    pred_spans_by_article = {
        aid: merge_overlapping_spans(spans)
        for aid, spans in pred_spans_by_article_raw.items()
    }

    token_p, token_r, token_f1 = token_level_f1(all_gold_token, all_pred_token)
    return pred_spans_by_article, (token_p, token_r, token_f1)

def build_gold_spans_by_article(article_records):
    d = {}
    for rec in article_records:
        d[rec["article_id"]] = rec["gold_spans"]
    return d


def train_one_epoch(model, loader, optimizer, scheduler, device):
    model.train()
    total_loss = 0.0

    for step, batch in enumerate(tqdm(loader, desc="Training")):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        crf_mask = batch["crf_mask"].to(device)
        pos_ids = batch["pos_ids"].to(device)
        ner_ids = batch["ner_ids"].to(device)
        labels = batch["labels"].to(device)

        optimizer.zero_grad(set_to_none=True)

        # input checks
        if not torch.isfinite(input_ids.float()).all():
            raise ValueError(f"Non-finite input_ids at step {step}")
        if not torch.isfinite(attention_mask.float()).all():
            raise ValueError(f"Non-finite attention_mask at step {step}")
        if not torch.isfinite(pos_ids.float()).all():
            raise ValueError(f"Non-finite pos_ids at step {step}")
        if not torch.isfinite(ner_ids.float()).all():
            raise ValueError(f"Non-finite ner_ids at step {step}")

        loss = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            crf_mask=crf_mask,
            pos_ids=pos_ids,
            ner_ids=ner_ids,
            labels=labels
        )

        if not torch.isfinite(loss):
            print(f"Non-finite loss at step {step}")
            print("article_ids:", batch["article_id"])
            raise ValueError("NaN/Inf loss detected")

        loss.backward()

        for name, p in model.named_parameters():
            if p.grad is not None and not torch.isfinite(p.grad).all():
                raise ValueError(f"Non-finite gradient in {name} at step {step}")

        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        total_loss += loss.item()

    return total_loss / max(1, len(loader))

def evaluate(model, loader, article_records, device):
    gold_spans_by_article = build_gold_spans_by_article(article_records)
    pred_spans_by_article, (tok_p, tok_r, tok_f1) = predict_spans(model, loader, device)
    span_p, span_r, span_f1 = exact_span_f1(gold_spans_by_article, pred_spans_by_article)

    metrics = {
        "token_precision": tok_p,
        "token_recall": tok_r,
        "token_f1": tok_f1,
        "span_precision": span_p,
        "span_recall": span_r,
        "span_f1": span_f1,
    }
    return metrics, pred_spans_by_article


def run_training(train_data, val_data, cfg: CFG):
    set_seed(cfg.seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # -------- Build article-level records
    train_articles = build_article_records(train_data)
    val_articles = build_article_records(val_data)
    test_articles = build_article_records(test_data)

    # -------- Tokenizer
    tokenizer = AutoTokenizer.from_pretrained(cfg.model_name, use_fast=True)

    if cfg.pos_dim > 0 or cfg.ner_dim > 0:
        nlp = spacy.load("en_core_web_sm", disable=["lemmatizer", "textcat"])
        pos_vocab, ner_vocab = build_pos_ner_vocab(train_articles, nlp)
    else:
        nlp = None
        pos_vocab = {"<PAD>": 0}
        ner_vocab = {"<PAD>": 0}

    # -------- Datasets
    train_ds = PTCSpanDataset(
        train_articles, tokenizer, nlp, pos_vocab, ner_vocab,
        max_length=cfg.max_length, stride=cfg.stride, is_train=True
    )
    val_ds = PTCSpanDataset(
        val_articles, tokenizer, nlp, pos_vocab, ner_vocab,
        max_length=cfg.max_length, stride=cfg.stride, is_train=False
    )
    # test_ds = PTCSpanDataset(
    #     test_articles, tokenizer, nlp, pos_vocab, ner_vocab,
    #     max_length=cfg.max_length, stride=cfg.stride, is_train=False
    # )

    # -------- Loaders
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size, shuffle=True,
        num_workers=cfg.num_workers, collate_fn=collate_fn
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size, shuffle=False,
        num_workers=cfg.num_workers, collate_fn=collate_fn
    )
    # test_loader = DataLoader(
    #     test_ds, batch_size=cfg.batch_size, shuffle=False,
    #     num_workers=cfg.num_workers, collate_fn=collate_fn
    # )

    # -------- Model
    model = DebertaSpanTagger(
        model_name=cfg.model_name,
        num_labels=NUM_LABELS,
        num_pos_tags=len(pos_vocab),
        num_ner_tags=len(ner_vocab),
        pos_dim=cfg.pos_dim,
        ner_dim=cfg.ner_dim,
        discourse_dim=0,
        lstm_hidden=cfg.lstm_hidden,
        dropout=cfg.dropout
    ).to(device)

    model.float()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay
    )

    total_steps = cfg.epochs * len(train_loader)
    warmup_steps = int(cfg.warmup_ratio * total_steps)

    scheduler = get_linear_schedule_with_warmup(
        optimizer,
        num_warmup_steps=warmup_steps,
        num_training_steps=total_steps
    )

    scaler = torch.amp.GradScaler("cuda", enabled=(device.type == "cuda"))

    best_val_f1 = -1
    best_state = None

    # -------- Training loop
    for epoch in range(1, cfg.epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, scheduler, device)

        val_metrics, _ = evaluate(model, val_loader, val_articles, device)

        print(
            f"Epoch {epoch}/{cfg.epochs} | "
            f"train_loss={train_loss:.4f} | "
            f"val_token_f1={val_metrics['token_f1']:.4f} | "
            f"val_span_f1={val_metrics['span_f1']:.4f}"
        )

        if val_metrics["span_f1"] > best_val_f1:
            best_val_f1 = val_metrics["span_f1"]
            best_state = {
                "model_state_dict": model.state_dict(),
                "pos_vocab": pos_vocab,
                "ner_vocab": ner_vocab,
                "cfg": cfg.__dict__,
                "best_val_span_f1": best_val_f1,
            }
            torch.save(best_state, cfg.save_path)
            print(f"Saved best model to {cfg.save_path}")

    # # -------- Load best and test
    # ckpt = torch.load(cfg.save_path, map_location=device)
    # model.load_state_dict(ckpt["model_state_dict"])

    # test_metrics, test_pred_spans = evaluate(model, test_loader, test_articles, device)

    # print("\n===== TEST RESULTS =====")
    # for k, v in test_metrics.items():
    #     print(f"{k}: {v:.4f}")
    return model, tokenizer, pos_vocab, ner_vocab, train_ds, val_ds
    return model, tokenizer, pos_vocab, ner_vocab, test_pred_spans, train_ds, val_ds, test_ds




In [ ]:
cfg = CFG(
    model_name="roberta-large",
    max_length=512,
    stride=384,   # 128-token overlap
    batch_size=4,
    lr = 1e-5,
    epochs=5,
    save_path="best_span_roberta_no_pos_ner_2.pt"
)

model, tokenizer, pos_vocab, ner_vocab, train_ds, val_ds = run_training(
    train_data=train_data,
    val_data=val_data,
    cfg=cfg
)

Loading weights: 100%|██████████| 389/389 [00:00<00:00, 4972.80it/s]
RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
Predicting: 100%|██████████| 303/303 [00:38<00:00,  7.89it/s]


Epoch 1/5 | train_loss=0.5355 | val_token_f1=0.3649 | val_span_f1=0.0832
Saved best model to best_span_roberta_no_pos_ner.pt


Predicting: 100%|██████████| 303/303 [00:39<00:00,  7.58it/s]


Epoch 2/5 | train_loss=0.3814 | val_token_f1=0.5380 | val_span_f1=0.1671
Saved best model to best_span_roberta_no_pos_ner.pt


Predicting: 100%|██████████| 303/303 [00:40<00:00,  7.50it/s]


...

Epoch 4/5 | train_loss=0.2421 | val_token_f1=0.5423 | val_span_f1=0.2395
Saved best model to best_span_roberta_no_pos_ner.pt


Predicting: 100%|██████████| 303/303 [00:40<00:00,  7.51it/s]


Epoch 5/5 | train_loss=0.2019 | val_token_f1=0.5835 | val_span_f1=0.2640
Saved best model to best_span_roberta_no_pos_ner.pt


Predicting: 100%|██████████| 304/304 [00:40<00:00,  7.50it/s]



===== TEST RESULTS =====
token_precision: 0.6160
token_recall: 0.5415
token_f1: 0.5763
span_precision: 0.3415
span_recall: 0.2011
span_f1: 0.2531
